Now we're going to look at the case of the Kibble-Zurek quench in two dimensions. The model is the same:

$$\ddot \phi + \eta \dot \phi - \nabla^2 \phi + \frac{1}{2}(\phi^3-\epsilon \phi) = \zeta$$

With a random un-correlated noise kernel

$$\langle \zeta \zeta \rangle = 2\eta \theta \delta^2 ( \vec{r}-\vec{r}')\delta(t-t')$$


The system is quenched from $\epsilon =-1 \text{to} \epsilon=3$, very deep into the ordered regime. This quench is controlled by a finite velocity:

$\epsilon=t/\tau$ 

Which according to the Kibble-Zurek mechanism will cause the correlation length of the system to freeze-out at $\xi_{KZ}=\tau^r$. In this model, $r=1/4$.



![](../figures/2d/kz_quench.gif)


In [ ]:
# General Imports
import sys; sys.path.insert(0, "..")
import json
import numpy as np
import matplotlib.pyplot as plt
import src.analysis as an
import os

In [ ]:
# ---- inputs ----
RESULTS  = "../results/kz_scaling.json"
FONTSIZE = 16

r      = json.load(open(RESULTS))["results"]
taus   = np.array([x["tau"] for x in r])
ell    = np.array([x["ell"] for x in r])
xi_hat = np.array([x["xi_hat"] for x in r])
t_rat  = np.array([x["t_f"] / x["t_hat"] for x in r])

for tau, l, xh, tr in zip(taus, ell, xi_hat, t_rat):
    print(f"tau={tau:4d}  t_f/t_hat={tr:4.2f}  ell={l:6.2f}  ell/xi_hat={l / xh:4.2f}")

slope = np.polyfit(np.log(taus), np.log(ell), 1)[0]

plt.figure(figsize=(8, 6))
plt.plot(taus, ell, "o", ms=10, label=r"$\ell$ at formation")
plt.plot(taus, ell[0] * (taus / taus[0])**0.25, "r--", lw=2, label=r"KZ: $\tau_Q^{1/4}$")
plt.xscale("log"); plt.yscale("log")
plt.xlabel(r"$\tau_Q$", fontsize=FONTSIZE)
plt.ylabel(r"domain size $\ell$ at formation", fontsize=FONTSIZE)
plt.title(rf"fitted slope = {slope:.2f}  (KZ: 0.25)", fontsize=FONTSIZE)
plt.tick_params(labelsize=FONTSIZE - 2)
plt.legend(fontsize=FONTSIZE - 2)
plt.tight_layout()
plt.savefig("../figures/2d/kz_scaling.png", dpi=150)
plt.show()

In [ ]:
# ---- inputs ----
TAUS     = [8, 16, 32, 64, 128, 256]   # missing data is skipped
CHUNK    = 0
SAMPLE   = 0
SHOW_HAT = [2, 4, 5, 6, 8]            # snapshot times to show, in units of t_hat
DX       = 0.5
FONTSIZE = 16

# keep only taus with data on this machine
paths = {tau: f"../data/2D_v2/tau{tau}/chunk_{CHUNK:03d}.npz" for tau in TAUS}
taus  = [tau for tau in TAUS if os.path.exists(paths[tau])]
print("showing tau =", taus, "  skipped:", [tau for tau in TAUS if tau not in taus])

n_cols = len(SHOW_HAT) + 2            # snapshots + formation target + end target
fig, axes = plt.subplots(len(taus), n_cols, figsize=(3.2 * n_cols, 3.6 * len(taus)), squeeze=False)

for row, tau in enumerate(taus):
    d = np.load(paths[tau])
    t_hat, xi_hat, N = float(d["t_hat"]), float(d["xi_hat"]), int(d["N"])

    # sanity checks: formation time and KZ domain size in units of t_hat, xi_hat
    tf  = d["t_form"] / t_hat
    ell = (N * DX)**2 / an.wall_length(d["y_form"], dx=DX)
    print(f"tau={tau:4d}  t_form/t_hat = {np.nanmean(tf):.2f} ± {np.nanstd(tf):.2f}   "
          f"ell_form/xi_hat = {ell.mean() / xi_hat:.2f}   (n={len(tf)})")

    # snapshots (per-panel scale: amplitude grows by orders of magnitude)
    for col, s in enumerate(SHOW_HAT):
        i = int(np.argmin(np.abs(d["snap_hat"] - s)))
        f = d["snaps"][SAMPLE, i].astype(np.float32)
        v = np.abs(f).max()
        axes[row, col].imshow(f, cmap="RdBu_r", vmin=-v, vmax=v, interpolation="nearest")
        if row == 0:
            axes[row, col].set_title(rf"$t/\hat t = {d['snap_hat'][i]:.1f}$", fontsize=FONTSIZE)

    # targets
    axes[row, -2].imshow(d["y_form"][SAMPLE], cmap="RdBu_r", vmin=0, vmax=1, interpolation="nearest")
    axes[row, -1].imshow(d["y_end"][SAMPLE],  cmap="RdBu_r", vmin=0, vmax=1, interpolation="nearest")
    if row == 0:
        axes[row, -2].set_title("target: formation", fontsize=FONTSIZE)
        axes[row, -1].set_title(rf"target: $t = {float(d['t_max_hat']):.0f}\,\hat t$", fontsize=FONTSIZE)

    axes[row, 0].set_ylabel(rf"$\tau_Q = {tau}$", fontsize=FONTSIZE)
    for ax in axes[row]:
        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig("../figures/2d/representative_chunks.png", dpi=150)
plt.show()

In [ ]:
# ---- inputs ----
TAUS     = [8, 256]
CHUNK    = 0
SAMPLE   = 1
EARLY    = 2.0          # early snapshot time, in units of t_hat (before formation)
FONTSIZE = 16

fig, axes = plt.subplots(len(TAUS), 2, figsize=(11, 5.5 * len(TAUS)), squeeze=False)

for row, tau in enumerate(TAUS):
    d = np.load(f"../data/2D_v2/tau{tau}/chunk_{CHUNK:03d}.npz")
    i = int(np.argmin(np.abs(d["snap_hat"] - EARLY)))
    f      = d["snaps"][SAMPLE, i].astype(np.float32)
    y_form = d["y_form"][SAMPLE].astype(float)
    y_end  = d["y_end"][SAMPLE].astype(float)
    tf_hat = d["t_form"][SAMPLE] / float(d["t_hat"])

    # left: early field, its walls (black) vs formation walls (green)
    ax = axes[row, 0]
    v  = np.abs(f).max()
    ax.imshow(f, cmap="RdBu_r", vmin=-v, vmax=v, alpha=0.5, interpolation="nearest")
    ax.contour(f,      levels=[0],   colors="k",       linewidths=2)
    ax.contour(y_form, levels=[0.5], colors="#f80df8", linewidths=4, linestyles="--")
    ax.set_title(rf"$t = {d['snap_hat'][i]:.1f}\,\hat t$ (black) vs formation (green)", fontsize=FONTSIZE)

    # right: formation walls (black) vs end walls (green)
    ax = axes[row, 1]
    ax.imshow(y_form, cmap="RdBu_r", vmin=0, vmax=1, alpha=0.5, interpolation="nearest")
    ax.contour(y_form, levels=[0.5], colors="k",       linewidths=2)
    ax.contour(y_end,  levels=[0.5], colors="#f80df8", linewidths=4, linestyles="--")
    ax.set_title(rf"formation, $t = {tf_hat:.1f}\,\hat t$ (black) vs end (green)", fontsize=FONTSIZE)

    axes[row, 0].set_ylabel(rf"$\tau_Q = {tau}$", fontsize=FONTSIZE)
    for ax in axes[row]:
        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig("../figures/2d/walls_form_vs_end.png", dpi=150)
plt.show()